[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/05-task-scheduling.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/05-task-scheduling.ipynb)

# Module 3.5 — Task Scheduling
**Module 3: Automation & Scripting** | Estimated time: 30 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Schedule repeating tasks with the `schedule` library using human-readable syntax
- Use APScheduler's `BlockingScheduler` and `BackgroundScheduler`
- Configure `cron`, `interval`, and `date` triggers in APScheduler
- Run scheduled jobs in a background thread so the notebook stays responsive
- Understand cron syntax (minute / hour / day / month / weekday)
- Choose the right scheduling approach for your use case

In [ ]:
!pip install schedule apscheduler -q

import schedule
import time
import threading
import datetime
from apscheduler.schedulers.blocking import BlockingScheduler
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
from apscheduler.triggers.interval import IntervalTrigger
from apscheduler.triggers.date import DateTrigger

print('schedule version:', schedule.__version__)
print('APScheduler ready.')

## 1. Cron Syntax Primer

Cron expressions have 5 fields separated by spaces:

```
.----------- minute      (0 - 59)
|  .--------- hour        (0 - 23)
|  |  .------- day-of-month (1 - 31)
|  |  |  .----- month       (1 - 12 or JAN-DEC)
|  |  |  |  .--- day-of-week  (0 - 6, 0=Sunday, or MON-SUN)
|  |  |  |  |
*  *  *  *  *
```

Special characters: `*` (any), `,` (list), `-` (range), `/` (step)

| Expression | Meaning |
|---|---|
| `* * * * *` | Every minute |
| `0 9 * * 1-5` | 09:00 Mon-Fri |
| `30 18 * * 5` | 18:30 every Friday |
| `0 0 1 * *` | Midnight on the 1st of every month |
| `*/15 * * * *` | Every 15 minutes |
| `0 9,12,17 * * *` | At 09:00, 12:00, and 17:00 every day |

In [ ]:
# Helper: track job calls so we can verify scheduling in the notebook
job_log = []

def log(label):
    ts = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
    msg = f'[{ts}] {label}'
    job_log.append(msg)
    print(msg)

def send_report():
    log('send_report  -> daily report sent')

def backup_database():
    log('backup_db    -> backup completed')

def cleanup_temp():
    log('cleanup_temp -> /tmp cleared')

def health_check():
    log('health_check -> service OK')

print('Job functions defined.')

## 2. The `schedule` Library — Human-Readable Syntax

`schedule` is the simplest scheduler: you define jobs, then call `schedule.run_pending()` in a loop.

In [ ]:
# Clear any previously registered jobs
schedule.clear()

# Define jobs
schedule.every(2).seconds.do(health_check)          # every 2 seconds
schedule.every(10).seconds.do(cleanup_temp)         # every 10 seconds
schedule.every().hour.do(send_report)               # every hour
schedule.every().day.at('09:00').do(send_report)    # daily at 09:00
schedule.every().monday.do(backup_database)         # every Monday
schedule.every(5).minutes.do(health_check)          # every 5 minutes

print('Registered jobs:')
for job in schedule.jobs:
    print(f'  {str(job)}')

In [ ]:
# Run for 6 seconds to observe the 2-second health checks
print('Running schedule loop for 6 seconds...\n')
end_time = time.time() + 6
while time.time() < end_time:
    schedule.run_pending()
    time.sleep(0.5)

print(f'\nTotal job executions: {len(job_log)}')
schedule.clear()
print('Schedule cleared.')

## 3. APScheduler — BackgroundScheduler

`BackgroundScheduler` runs in a daemon thread automatically, so your main code keeps running. This is the most useful mode for notebooks and long-running applications.

In [ ]:
job_log.clear()

bg = BackgroundScheduler()

# Interval trigger
bg.add_job(health_check, IntervalTrigger(seconds=2), id='health', name='Health Check')

# Cron trigger — every minute at second 0 (won't fire in our short demo)
bg.add_job(send_report, CronTrigger(minute='*'), id='report', name='Send Report')

# Date trigger — run once at a specific datetime (5 seconds from now)
run_at = datetime.datetime.now() + datetime.timedelta(seconds=5)
bg.add_job(backup_database, DateTrigger(run_date=run_at), id='backup', name='Backup DB')

bg.start()
print('BackgroundScheduler started. Running for 8 seconds...\n')
time.sleep(8)

print(f'\nJobs executed ({len(job_log)} total):')
for entry in job_log:
    print(' ', entry)

bg.shutdown(wait=False)
print('\nBackgroundScheduler stopped.')

## 4. APScheduler — BlockingScheduler (Script Pattern)

`BlockingScheduler` takes over the current thread. It is designed for standalone scripts where the scheduler IS the main program. Here we run it in a thread to keep the notebook cell from hanging.

In [ ]:
job_log.clear()

_bs_instance = None

def run_blocking_scheduler():
    global _bs_instance
    _bs_instance = BlockingScheduler()
    _bs_instance.add_job(health_check, 'interval', seconds=2, id='hc')
    _bs_instance.add_job(cleanup_temp, 'interval', seconds=5, id='ct')
    _bs_instance.start()  # blocks here

thread = threading.Thread(target=run_blocking_scheduler, daemon=True)
thread.start()
time.sleep(7)

if _bs_instance:
    _bs_instance.shutdown(wait=False)

print(f'Jobs executed in 7 seconds: {len(job_log)}')
for entry in job_log:
    print(' ', entry)

## 5. CronTrigger Examples

In [ ]:
# Demonstrate next fire times for various cron expressions
examples = [
    ('Weekdays 09:00',       CronTrigger(hour=9, minute=0, day_of_week='mon-fri')),
    ('Every 15 minutes',     CronTrigger(minute='*/15')),
    ('1st of month midnight',CronTrigger(day=1, hour=0, minute=0)),
    ('Fridays 18:30',        CronTrigger(hour=18, minute=30, day_of_week='fri')),
    ('9, 12, 17 daily',      CronTrigger(hour='9,12,17', minute=0)),
]

now = datetime.datetime.now(datetime.timezone.utc)
print(f'Current UTC time: {now.strftime("%Y-%m-%d %H:%M:%S")}\n')
print(f'{"Description":30s}  Next fire time')
print('-' * 65)
for desc, trigger in examples:
    nxt = trigger.get_next_fire_time(None, now)
    nxt_str = nxt.strftime('%Y-%m-%d %H:%M:%S %Z') if nxt else 'N/A'
    print(f'{desc:30s}  {nxt_str}')

## 6. Running a Repeating Background Task with `threading`

For the simplest case — a single repeating task — you can skip the scheduler libraries entirely.

In [ ]:
def run_every(seconds, func, stop_event):
    """Call func every `seconds` seconds until stop_event is set."""
    while not stop_event.is_set():
        func()
        stop_event.wait(seconds)  # interruptible sleep

stop = threading.Event()

poll_thread = threading.Thread(
    target=run_every,
    args=(2, health_check, stop),
    daemon=True,
    name='HealthPoller',
)

print('Starting background health poller...')
job_log.clear()
poll_thread.start()
time.sleep(7)
stop.set()
poll_thread.join(timeout=2)

print(f'\nPoller ran {len(job_log)} times in ~7 seconds.')
for entry in job_log:
    print(' ', entry)

## 7. Choosing the Right Approach

| Tool | Best for | Pros | Cons |
|---|---|---|---|
| `schedule` | Simple scripts | Readable, zero config | Blocking loop, no persistence |
| `APScheduler BackgroundScheduler` | Long-running apps | Non-blocking, flexible triggers | More setup |
| `APScheduler BlockingScheduler` | Dedicated scheduler processes | Clear ownership | Blocks the thread |
| `threading` | Single repeating task | Lightweight, built-in | Manual stop logic |
| `cron` (system) | Production servers | Reliable, OS-managed | Not Python-native |
| `Celery` | Distributed task queues | Scalable, persistent | Heavy dependency |

## Practice Exercises

**Exercise 1 — Log Rotation Scheduler**  
Using the `schedule` library, write a script that:
1. Appends a timestamped line to `/tmp/app.log` every 3 seconds
2. Every 15 seconds, reads the log file, keeps only the last 5 entries, and writes them back (simulating log rotation)
Run it for 30 seconds and print the final log contents.

**Exercise 2 — APScheduler Job with Arguments**  
`add_job` accepts `args` and `kwargs` parameters. Create a `BackgroundScheduler` that runs a `ping(host)` function for three different hostnames every 4 seconds. `ping` should print the hostname and a simulated round-trip time (`random.uniform(1, 100)` ms). Run for 10 seconds.

**Exercise 3 — Cron Expression Inspector**  
Write a function `next_n_runs(cron_expr: str, n: int = 5) -> list` that parses a cron expression string (e.g. `'0 9 * * 1-5'`) using APScheduler's `CronTrigger.from_crontab()` and returns the next `n` fire times as formatted strings. Test with three different expressions.